# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

**Nombre:** Jose Armando Sarango Cuenca  
**Fecha:** 4/05/2026

### Paso previo
Descargar los 1000 libros

In [ ]:
import os
import requests
import time

def descargar_gutenberg_1000(cantidad=1000, carpeta_destino='gutenberg_1000'):
    if not os.path.exists(carpeta_destino):
        os.makedirs(carpeta_destino)

    libros_descargados = 0
    id_actual = 1

    print(f"Iniciando descarga de {cantidad} libros...")

    while libros_descargados < cantidad:
        url = f"https://www.gutenberg.org/cache/epub/{id_actual}/pg{id_actual}.txt"
        
        try:
            response = requests.get(url, timeout=10)
            
            if response.status_code == 200:
                nombre_archivo = os.path.join(carpeta_destino, f"libro_{id_actual}.txt")
                with open(nombre_archivo, 'w', encoding='utf-8') as f:
                    f.write(response.text)
                
                libros_descargados += 1
                if libros_descargados % 10 == 0:
                    print(f"Progreso: {libros_descargados}/{cantidad} libros descargados.")
                
                # Respeto al servidor: pausa breve entre descargas
                time.sleep(0.5) 
            else:
                pass
                
        except Exception as e:
            print(f"Error descargando ID {id_actual}: {e}")
        
        id_actual += 1
        
        # Seguridad para no entrar en bucle infinito si hay fallos de red
        if id_actual > 5000 and libros_descargados < 10:
            print("Demasiados fallos de conexión. Abortando.")
            break

    print(f"Proceso finalizado. Libros guardados en: {carpeta_destino}")

if __name__ == "__main__":
    descargar_gutenberg_1000(1000)


Iniciando descarga de 1000 libros...
Progreso: 10/1000 libros descargados.
Progreso: 20/1000 libros descargados.
Progreso: 30/1000 libros descargados.
Progreso: 40/1000 libros descargados.
Progreso: 50/1000 libros descargados.
Progreso: 60/1000 libros descargados.
Progreso: 70/1000 libros descargados.
Progreso: 80/1000 libros descargados.
Progreso: 90/1000 libros descargados.
Progreso: 100/1000 libros descargados.
Progreso: 110/1000 libros descargados.
Progreso: 120/1000 libros descargados.
Progreso: 130/1000 libros descargados.
Progreso: 140/1000 libros descargados.
Progreso: 150/1000 libros descargados.
Progreso: 160/1000 libros descargados.
Progreso: 170/1000 libros descargados.
Progreso: 180/1000 libros descargados.
Progreso: 190/1000 libros descargados.
Progreso: 200/1000 libros descargados.
Progreso: 210/1000 libros descargados.
Progreso: 220/1000 libros descargados.
Progreso: 230/1000 libros descargados.
Progreso: 240/1000 libros descargados.
Progreso: 250/1000 libros descargado

In [11]:
%pip install scikit-learn pandas numpy

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.0 MB 703.4 kB/s eta 0:00:11
   -- ------------------------------------- 0.5/8.0 MB 703.4 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.0 MB 704.6 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.0 MB 704.6 kB/s eta 0:00:11
   ----- ---------------------------------- 1.0/8.0 MB 705.0 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.0 MB 704.7 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.0 MB 782.3 kB/s eta 0:00:09
   ------- -------------------------------- 1.6/8.0 MB 782.3 kB/s eta 0:00:09
   --------- ------------------------------ 1.8/8.0 MB 768.9 kB/s eta 0:00:09
   ---------- --------


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importaciones

In [4]:
import os
import math
import re
import time
import numpy as np
from collections import Counter
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
print("Importaciones completadas.")

Importaciones completadas.


### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [5]:
turismo_path="C:\\Users\\ASUS ROG G17\\Documents\\RI\\Nueva carpeta\\ir26a-main\\ir26a-main\\03tfidf\\data\\01_corpus_turismo_500.txt"
with open(turismo_path, "r", encoding="utf-8") as f:
    documentos_turismo = [linea.strip() for linea in f if linea.strip()]

In [6]:
# TfidfVectorizer es una herramienta de sklearn que separa las palabras, calcula TF e IDF, y devuelve una matriz dispersa 
# con los pesos TF-IDF de cada término en cada documento.
vectorizer_turismo = TfidfVectorizer(
    sublinear_tf=True,   #aplica la fórmula: tf = 1 + log(frecuencia)
    norm='l2',           #normaliza cada vector dividiendo por su magnitud
    min_df=2             #ignora palabras que aparecen en menos de 2 documentos
)

matriz_turismo = vectorizer_turismo.fit_transform(documentos_turismo)
vocabulario_turismo = vectorizer_turismo.get_feature_names_out()

print(f"Dimensiones de la matriz TF-IDF: {matriz_turismo.shape}")
print(f"  → {matriz_turismo.shape[0]} documentos × {matriz_turismo.shape[1]} términos únicos")
print(f"\nPrimeros 20 términos del vocabulario: {vocabulario_turismo[:20]}")

Dimensiones de la matriz TF-IDF: (500, 116)
  → 500 documentos × 116 términos únicos

Primeros 20 términos del vocabulario: ['2000' 'agua' 'amazonía' 'arquitectura' 'artesanía' 'atrae' 'atraen'
 'auténtico' 'aventura' 'aves' 'avistamiento' 'baños' 'biodiversidad'
 'cajas' 'caminatas' 'canopy' 'centro' 'colonial' 'color' 'como']


In [7]:
# Mostramos una porción pequeña de la matriz como tabla para visualizarla.
# 0.0 = la palabra no aparece en ese documento, otro valor = sí aparece y ese es su peso.
df_turismo = pd.DataFrame(
    matriz_turismo[:5, :10].toarray(),
    columns=vocabulario_turismo[:10]
)
df_turismo.index.name = "doc_id"
print("Vista parcial de la matriz TF-IDF (5 docs × 10 términos):")
print(df_turismo.round(4))

Vista parcial de la matriz TF-IDF (5 docs × 10 términos):
        2000  agua  amazonía  arquitectura  artesanía  atrae  atraen  \
doc_id                                                                 
0        0.0   0.0       0.0           0.0     0.3493  0.000     0.0   
1        0.0   0.0       0.0           0.0     0.0000  0.000     0.0   
2        0.0   0.0       0.0           0.0     0.0000  0.353     0.0   
3        0.0   0.0       0.0           0.0     0.0000  0.000     0.0   
4        0.0   0.0       0.0           0.0     0.0000  0.000     0.0   

        auténtico  aventura  aves  
doc_id                             
0             0.0       0.0   0.0  
1             0.0       0.0   0.0  
2             0.0       0.0   0.0  
3             0.0       0.0   0.0  
4             0.0       0.0   0.0  


### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [8]:
path = "C:\\Users\\ASUS ROG G17\\Documents\\RI\\Nueva carpeta\\ir26a-main\\ir26a-main\\03tfidf\\data_1000"

contenidos = {}

for archivo in os.listdir(path):
    if archivo.endswith(".txt"):
        try:
            with open(os.path.join(path, archivo), "r", encoding="utf-8") as f:
                contenidos[archivo] = f.read()
        except:
            with open(os.path.join(path, archivo), "r", encoding="latin-1") as f:
                contenidos[archivo] = f.read()

nombres_docs = sorted(contenidos.keys())
corpus_textos = [contenidos[nombre] for nombre in nombres_docs]

print(f"Total de documentos leídos: {len(nombres_docs)}")

Total de documentos leídos: 1000


### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [10]:
# Limpiamos el texto: todo a minúsculas y solo letras (sin números ni puntuación).

def preprocesar(texto):
    texto = texto.lower()
    palabras = re.findall(r'[a-z]+', texto)
    return palabras
# Para cada libro contamos cuántas veces aparece cada palabra (eso es el TF).
print("Calculando TF para cada documento...")
inicio = time.time()

tf_docs = []
for i, texto in enumerate(corpus_textos):
    palabras = preprocesar(texto)
    tf_docs.append(Counter(palabras))# Counter cuenta las repeticiones de cada palabra
    if (i + 1) % 100 == 0:
        print(f"  Procesados {i+1}/1000 documentos...")

print(f"TF calculado en {time.time()-inicio:.1f}s")

Calculando TF para cada documento...
  Procesados 100/1000 documentos...
  Procesados 200/1000 documentos...
  Procesados 300/1000 documentos...
  Procesados 400/1000 documentos...
  Procesados 500/1000 documentos...
  Procesados 600/1000 documentos...
  Procesados 700/1000 documentos...
  Procesados 800/1000 documentos...
  Procesados 900/1000 documentos...
  Procesados 1000/1000 documentos...
TF calculado en 51.6s


In [11]:
# Juntamos todas las palabras de todos los libros en un solo vocabulario global.
N = len(corpus_textos)

vocabulario = set()
for tf in tf_docs:
    vocabulario.update(tf.keys())
vocabulario = sorted(vocabulario)
term2idx = {t: i for i, t in enumerate(vocabulario)}

print(f"Vocabulario total: {len(vocabulario):,} términos únicos")
# Contamos en cuántos documentos aparece cada palabra (eso es el DF).
df = Counter()
for tf in tf_docs:
    for termino in tf:
        df[termino] += 1
# Calculamos IDF = log(N / df): palabras comunes tienen IDF bajo, palabras raras tienen IDF alto.

idf = {}
for termino in vocabulario:
    idf[termino] = math.log(N / df[termino]) if df[termino] > 0 else 0.0

ejemplos = sorted(idf.items(), key=lambda x: x[1])
print("\n10 términos con IDF más BAJO (aparecen en casi todos los docs):")
for t, v in ejemplos[:10]:
    print(f"  '{t}': IDF={v:.4f} (en {df[t]}/{N} docs)")

print("\n10 términos con IDF más ALTO (muy raros):")
for t, v in ejemplos[-10:]:
    print(f"  '{t}': IDF={v:.4f} (en {df[t]}/{N} docs)")

Vocabulario total: 443,430 términos únicos

10 términos con IDF más BAJO (aparecen en casi todos los docs):
  'a': IDF=0.0000 (en 1000/1000 docs)
  'abide': IDF=0.0000 (en 1000/1000 docs)
  'about': IDF=0.0000 (en 1000/1000 docs)
  'accept': IDF=0.0000 (en 1000/1000 docs)
  'accepted': IDF=0.0000 (en 1000/1000 docs)
  'accepting': IDF=0.0000 (en 1000/1000 docs)
  'access': IDF=0.0000 (en 1000/1000 docs)
  'accessed': IDF=0.0000 (en 1000/1000 docs)
  'accessible': IDF=0.0000 (en 1000/1000 docs)
  'accordance': IDF=0.0000 (en 1000/1000 docs)

10 términos con IDF más ALTO (muy raros):
  'zzan': IDF=6.9078 (en 1/1000 docs)
  'zzfn': IDF=6.9078 (en 1/1000 docs)
  'zzzing': IDF=6.9078 (en 1/1000 docs)
  'zzzz': IDF=6.9078 (en 1/1000 docs)
  'zzzzed': IDF=6.9078 (en 1/1000 docs)
  'zzzzzzz': IDF=6.9078 (en 1/1000 docs)
  'zzzzzzzz': IDF=6.9078 (en 1/1000 docs)
  'zzzzzzzzz': IDF=6.9078 (en 1/1000 docs)
  'zzzzzzzzzz': IDF=6.9078 (en 1/1000 docs)
  'zzzzzzzzzzzzzzz': IDF=6.9078 (en 1/1000 docs

In [12]:
# Combinamos TF e IDF para obtener el peso final de cada palabra en cada documento.
# peso = (1 + log(frecuencia)) × idf  →  palabras frecuentes Y raras tienen mayor peso.
# Si el peso es 0 (palabra común en todo el corpus) simplemente no se guarda.
print("Construyendo matriz TF-IDF dispersa...")
inicio = time.time()

tfidf_docs = []

for i, tf_doc in enumerate(tf_docs):
    vector = {}
    for termino, freq in tf_doc.items():
        tf_val = 1 + math.log(freq)   
        idf_val = idf[termino]
        peso = tf_val * idf_val
        if peso > 0:                  
            vector[termino] = peso
    tfidf_docs.append(vector)

print(f"Matriz construida en {time.time()-inicio:.1f}s")

# Mostramos las palabras más representativas del primer libro como ejemplo.
top10 = sorted(tfidf_docs[0].items(), key=lambda x: x[1], reverse=True)[:10]
print(f"\nTop 10 términos TF-IDF de '{nombres_docs[0]}':")
for t, w in top10:
    print(f"  '{t}': {w:.4f}")

Construyendo matriz TF-IDF dispersa...
Matriz construida en 9.1s

Top 10 términos TF-IDF de 'libro_1.txt':
  'facsimiles': 9.1139
  'usurpations': 7.6592
  'etexts': 7.6359
  'brittish': 6.9078
  'diskpack': 6.2146
  'capitalizations': 6.2146
  'abolishing': 5.8354
  'comparably': 5.8091
  'sharewared': 5.8091
  'emailed': 5.5215


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [13]:
# Mide qué tan parecidos son dos vectores usando la fórmula del coseno.
# Devuelve un valor entre 0 (nada parecidos) y 1 (idénticos).
def similitud_coseno(vec_q, vec_d):
    dot = sum(vec_q[t] * vec_d.get(t, 0.0) for t in vec_q)
    norma_q = math.sqrt(sum(v**2 for v in vec_q.values()))
    norma_d = math.sqrt(sum(v**2 for v in vec_d.values()))
    if norma_q == 0 or norma_d == 0:
        return 0.0
    return dot / (norma_q * norma_d)


# Convierte el texto de la consulta en un vector TF-IDF,
# igual que se hizo con los documentos, para poder compararlos.
def vector_consulta(query_texto):
    palabras = preprocesar(query_texto)
    tf_query = Counter(palabras)
    vec = {}
    for termino, freq in tf_query.items():
        if termino in idf:
            tf_val = 1 + math.log(freq)
            idf_val = idf[termino]
            peso = tf_val * idf_val
            if peso > 0:
                vec[termino] = peso
    return vec


# Busca los documentos más relevantes para la consulta.
# Compara la consulta contra todos los libros y devuelve los top_k más similares.
def buscar(query, top_k=10):
    vec_q = vector_consulta(query)
    if not vec_q:
        print(f"Ningún término de '{query}' está en el vocabulario.")
        return []
    scores = []
    for i, vec_d in enumerate(tfidf_docs):
        sim = similitud_coseno(vec_q, vec_d)
        if sim > 0:
            scores.append((nombres_docs[i], sim))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

print("Función buscar() lista.")

Función buscar() lista.


In [14]:
# Probamos la función buscar() con dos consultas de ejemplo.
# El resultado muestra los libros más relevantes ordenados por similitud (mayor = más relevante).

inicio = time.time()
resultados = buscar("whale ocean sea")
tiempo = (time.time() - inicio) * 1000

print(f"Consulta: 'whale ocean sea'")
print(f"Tiempo: {tiempo:.1f}ms")
print(f"\nTop 10 documentos más relevantes:")
for i, (doc, sim) in enumerate(resultados, 1):
    print(f"  {i}. {doc:<30} similitud: {sim:.4f}")

print()

inicio = time.time()
resultados2 = buscar("love marriage family")
tiempo2 = (time.time() - inicio) * 1000

print(f"Consulta: 'love marriage family'")
print(f"Tiempo: {tiempo2:.1f}ms")
print(f"\nTop 10 documentos más relevantes:")
for i, (doc, sim) in enumerate(resultados2, 1):
    print(f"  {i}. {doc:<30} similitud: {sim:.4f}")

Consulta: 'whale ocean sea'
Tiempo: 1013.5ms

Top 10 documentos más relevantes:
  1. libro_473.txt                  similitud: 0.0359
  2. libro_321.txt                  similitud: 0.0316
  3. libro_164.txt                  similitud: 0.0277
  4. libro_15.txt                   similitud: 0.0273
  5. libro_641.txt                  similitud: 0.0256
  6. libro_1018.txt                 similitud: 0.0246
  7. libro_640.txt                  similitud: 0.0245
  8. libro_585.txt                  similitud: 0.0242
  9. libro_393.txt                  similitud: 0.0238
  10. libro_412.txt                  similitud: 0.0225

Consulta: 'love marriage family'
Tiempo: 997.1ms

Top 10 documentos más relevantes:
  1. libro_946.txt                  similitud: 0.0183
  2. libro_927.txt                  similitud: 0.0170
  3. libro_57.txt                   similitud: 0.0168
  4. libro_856.txt                  similitud: 0.0166
  5. libro_398.txt                  similitud: 0.0162
  6. libro_340.txt      